# Spark Auto-Configurator
### A self-configuring utility for PySpark / Microsoft Fabric notebooks

**What this notebook builds:** a small library (`SparkAutoConfigurator`) that looks at the
data you're about to process and the cluster you're about to process it on, and produces a
**complete, cited, ready-to-apply set of Spark configuration values** — shuffle partitions,
executor shape, memory fractions, broadcast threshold, and (on Fabric) Efficient Scaledown —
instead of you carrying those formulas around in your head or copy-pasting them from a blog
post every time.

**Companion document:** this notebook assumes familiarity with *"Efficient Scaledown & the
Remote Shuffle Manager — Internals Reference"* and the *"Spark Internals"* interactive guide
produced alongside it. Where a recommendation below is a Fabric-specific behaviour, it cites
back to the relevant Part of that document rather than re-explaining it.

---

## How to read every recommendation this notebook produces

Every setting comes with a **basis** tag. This notebook will never dress a rule of thumb up as
a hard fact:

| Tag | Meaning |
|---|---|
| `SPARK_DEFAULT` | A documented default shipped by Apache Spark itself. |
| `FABRIC_DOC` | A figure or behaviour Microsoft documents for Fabric specifically. |
| `HEURISTIC` | A widely-used community/industry rule of thumb — a **starting point to validate against the Spark UI**, not a guarantee. |

Every setting also comes with a **scope** tag, and this one is not cosmetic — it was found by
actually breaking this notebook against a live Spark session (Part 2 below):

| Tag | Meaning |
|---|---|
| `runtime` | Safe to change on a session that's already running, via `spark.conf.set(...)`. |
| `session_start` | Must be set **before** the session starts (executor cores/memory, memory fractions, dynamic allocation, shuffle-plugin wiring). Calling `spark.conf.set()` on one of these raises `AnalysisException: [CANNOT_MODIFY_CONFIG]` — this notebook found that out the hard way against a real local Spark session and built the fix in, rather than asserting it from memory. |

## What this notebook is *not*

It does not know your query plan. Its shuffle-size estimate is a proxy based on **input** data
volume, not a measurement of actual shuffle write bytes (which depends on the operation — a
`groupBy` that aggregates away 95% of rows shuffles far less than a `join` that explodes rows).
Treat every `HEURISTIC`-tagged value as the number to start from, then correct it against the
Spark UI's actual stage metrics once the job has run once. Part 8 spells this out in full.

## Part 0 — Environment check

Detects whether this is running inside a Fabric notebook (an injected `notebookutils` module
is the signal — a convenience heuristic, not a documented detection API) or standalone, and
reports the PySpark version in use.

In [1]:
import sys
print(f"Python {sys.version.split()[0]}")

try:
    import pyspark
    print(f"PySpark {pyspark.__version__}")
except ImportError:
    print("PySpark not importable in this kernel — install it, or run this notebook inside a Fabric/Databricks session.")

try:
    import notebookutils  # noqa: F401
    PLATFORM = "fabric"
    print("notebookutils detected → assuming Fabric notebook environment.")
except ImportError:
    PLATFORM = "oss"
    print("notebookutils not found → assuming standalone/OSS Spark environment.")

Python 3.12.3


PySpark 3.5.1
notebookutils not found → assuming standalone/OSS Spark environment.


---
## Part 1 — The heuristic engine (`spark_autoconfig_core`)

Pure Python, **zero PySpark dependency**, so it can be unit-tested without a live cluster and
reused by other tooling (the interactive HTML "Spark Internals" doc's Config Advisor panel
runs a JavaScript port of the same formulas, so the two stay consistent).

Save this cell's content as `spark_autoconfig_core.py` next to this notebook to import it
normally elsewhere (`from spark_autoconfig_core import SparkAutoConfigurator` — see Part 2);
it's inlined here so this notebook is runnable standalone, top to bottom, with no setup step.

In [2]:
"""
spark_autoconfig_core.py
=========================
Pure-Python heuristic engine behind the Spark Auto-Configurator notebook utility.

Deliberately has ZERO dependency on `pyspark` or `notebookutils` in this module, so every
heuristic can be unit-tested in any Python interpreter, independent of a live Spark session.
The notebook (spark_auto_config_utility.ipynb) imports this module and wraps it with the
Spark- and Fabric-facing I/O (reading paths, calling spark.conf.set, etc).

Every recommendation below cites its basis. Three categories are used throughout, and every
`Recommendation` records which one applies:

  SPARK_DEFAULT   - a documented default shipped by Apache Spark itself.
  FABRIC_DOC      - a figure or behaviour documented by Microsoft for Fabric specifically.
  HEURISTIC       - a widely-used community/industry rule of thumb, not a hard Spark rule.
                     These are starting points to validate against the Spark UI, not guarantees.

This distinction matters: this module will never present a HEURISTIC as if it were a
SPARK_DEFAULT or a documented Fabric behaviour.
"""

from dataclasses import dataclass, field
from typing import Optional, List, Dict, Literal
import math

Basis = Literal["SPARK_DEFAULT", "FABRIC_DOC", "HEURISTIC"]
Scope = Literal["runtime", "session_start"]

# --------------------------------------------------------------------------------------
# Config mutability. Empirically verified against a live local Spark 3.5.1 session
# (see test_e2e.py / probe_mutability.py in the companion utility repo): calling
# spark.conf.set() on a "session_start" key raises AnalysisException[CANNOT_MODIFY_CONFIG].
# These configs are read once when the executor/driver JVM (SparkEnv) starts, so they must
# be supplied before the session is created — via spark-submit / SparkConf on OSS Spark, or
# via the %%configure magic / environment Compute pane on Fabric. This distinction is real
# and load-bearing: a notebook that calls spark.conf.set("spark.executor.cores", ...) after
# the session already exists will simply crash, not silently no-op.
SESSION_START_PREFIXES = (
    "spark.executor.cores", "spark.executor.memory", "spark.executor.instances",
    "spark.driver.memory", "spark.driver.cores",
    "spark.memory.fraction", "spark.memory.storageFraction", "spark.memory.offHeap",
    "spark.dynamicAllocation.",
    # Shuffle-manager/plugin wiring (RSM, Shuffle Migration, Decision Layer) is resolved when
    # SparkEnv is constructed — architecturally a session-start decision even where a given
    # Spark build doesn't register the key as formally immutable. Treat as session-start.
    "spark.remote.shuffle.", "spark.storage.decommission.", "spark.sql.rsm.",
)


def _scope_for_key(key: str) -> Scope:
    if key.startswith("_"):
        return "runtime"  # informational notes; not a real config
    return "session_start" if key.startswith(SESSION_START_PREFIXES) else "runtime"



# --------------------------------------------------------------------------------------
# Reference constants (all cited in the notebook's markdown; kept here as single source
# of truth so the notebook and the interactive HTML doc can both be generated from them).
# --------------------------------------------------------------------------------------

SPARK_DEFAULTS = {
    "spark.sql.files.maxPartitionBytes": "134217728",          # 128 MB
    "spark.sql.shuffle.partitions": "200",
    "spark.sql.adaptive.enabled": "true",                      # since Spark 3.2
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes": "67108864",  # 64 MB
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1048576",  # 1 MB
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor": "5",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes": "268435456",  # 256 MB
    "spark.sql.autoBroadcastJoinThreshold": "10485760",        # 10 MB
    "spark.memory.fraction": "0.6",
    "spark.memory.storageFraction": "0.5",
    "spark.executor.memoryOverhead_factor": "0.10",            # min 384MB
    "spark.dynamicAllocation.executorIdleTimeout": "60s",
    "spark.dynamicAllocation.schedulerBacklogTimeout": "1s",   # 1s in modern Spark (was 5s pre-2.x docs vary; validate per version)
}

# Fabric Spark pool node sizes are memory-optimized at a fixed 8 GB of RAM per vCore.
# Source: Microsoft Learn, "Apache Spark compute for Data Engineering and Data Science".
FABRIC_NODE_SIZES = [
    {"name": "Small",   "vcores": 4,  "memory_gb": 32},
    {"name": "Medium",  "vcores": 8,  "memory_gb": 64},
    {"name": "Large",   "vcores": 16, "memory_gb": 128},
    {"name": "XLarge",  "vcores": 32, "memory_gb": 256},
    {"name": "XXLarge", "vcores": 64, "memory_gb": 512},
]

# 1 Fabric Capacity Unit (CU) = 2 Spark vCores. Source: Microsoft Learn, same page.
FABRIC_VCORES_PER_CU = 2


@dataclass
class Recommendation:
    key: str
    value: str
    basis: Basis
    reason: str
    scope: Scope = field(init=False)

    def __post_init__(self):
        self.scope = _scope_for_key(self.key)


@dataclass
class DataProfile:
    total_bytes: int
    file_count: int
    avg_file_size_bytes: Optional[int] = None
    estimated_row_count: Optional[int] = None
    smallest_join_side_bytes: Optional[int] = None
    expected_skew: bool = False
    source_format: str = "delta"

    def __post_init__(self):
        if self.avg_file_size_bytes is None and self.file_count > 0:
            self.avg_file_size_bytes = int(self.total_bytes / self.file_count)

    @property
    def total_gb(self) -> float:
        return self.total_bytes / (1024 ** 3)


@dataclass
class ClusterProfile:
    platform: Literal["fabric", "oss"] = "fabric"
    node_size: Optional[str] = None       # Fabric only, e.g. "Medium"
    num_nodes: int = 1                    # includes the driver node
    node_vcores: Optional[int] = None     # generic/OSS: vCores per worker node
    node_memory_gb: Optional[float] = None
    driver_memory_gb: Optional[float] = None

    def __post_init__(self):
        if self.platform == "fabric" and self.node_size:
            spec = next((n for n in FABRIC_NODE_SIZES if n["name"] == self.node_size), None)
            if spec:
                self.node_vcores = spec["vcores"]
                self.node_memory_gb = spec["memory_gb"]

    @property
    def num_executors(self) -> int:
        """Fabric: node:executor ratio is always 1:1, one node reserved for the driver
        (except single-node pools, where driver and executor share the one node)."""
        if self.platform == "fabric":
            return 1 if self.num_nodes <= 1 else self.num_nodes - 1
        # Generic/OSS: unknown executor-per-node packing without more info; assume 1:1
        # unless the caller overrides — conservative default, flagged in the report.
        return max(1, self.num_nodes - 1)

    @property
    def total_executor_vcores(self) -> int:
        return self.num_executors * (self.node_vcores or 0)


@dataclass
class WorkloadHints:
    kind: Literal["batch_etl", "interactive", "ml_training", "streaming"] = "batch_etl"
    cache_heavy: bool = False
    many_small_joins: bool = False


# --------------------------------------------------------------------------------------
# Individual heuristic functions — each one independently testable.
# --------------------------------------------------------------------------------------

def nearest_fabric_node(required_vcores: int) -> dict:
    """Smallest Fabric node size whose vCore count meets or exceeds the requirement."""
    for spec in FABRIC_NODE_SIZES:
        if spec["vcores"] >= required_vcores:
            return spec
    return FABRIC_NODE_SIZES[-1]


def estimate_shuffle_partitions(
    total_input_bytes: int,
    total_executor_vcores: int,
    target_partition_mb: int = 128,
    shuffle_fraction_of_input: float = 1.0,
) -> Recommendation:
    """
    Target: land close to `target_partition_mb` per shuffle partition (this mirrors the
    128 MB default Spark already uses for file-scan partitioning via
    spark.sql.files.maxPartitionBytes, applied here to the shuffle side too).

    `shuffle_fraction_of_input` is an explicit fudge factor: a shuffle's actual output size
    depends on the operation (a groupBy that aggregates away 90% of rows shuffles far less
    than its input; a join that explodes rows shuffles more). Without knowing the query,
    input size is used as the proxy — this is a HEURISTIC starting point, not a measurement.
    Validate and correct against the Spark UI's actual "Shuffle Write" metric after a real run.
    """
    shuffle_bytes = int(total_input_bytes * shuffle_fraction_of_input)
    target_bytes = target_partition_mb * 1024 * 1024
    by_size = math.ceil(shuffle_bytes / target_bytes) if shuffle_bytes > 0 else 1
    # Parallelism floor: don't go below ~2x total executor cores, or every core can't stay busy
    by_parallelism = max(total_executor_vcores * 2, 1)
    partitions = max(by_size, by_parallelism)
    # Sanity ceiling — beyond this, scheduling overhead of tiny tasks dominates
    partitions = min(partitions, 200_000)
    reason = (
        f"max(size-based [{shuffle_bytes/1e9:.2f} GB \u00f7 {target_partition_mb} MB \u2192 {by_size}], "
        f"parallelism floor [{total_executor_vcores} vCores \u00d7 2 \u2192 {by_parallelism}])"
    )
    return Recommendation("spark.sql.shuffle.partitions", str(partitions), "HEURISTIC", reason)


def recommend_advisory_partition_size(target_partition_mb: int = 128) -> List[Recommendation]:
    target_bytes = target_partition_mb * 1024 * 1024
    return [
        Recommendation("spark.sql.adaptive.enabled", "true", "SPARK_DEFAULT",
                        "On by default since Spark 3.2 / all current Fabric runtimes; set explicitly for clarity."),
        Recommendation("spark.sql.adaptive.coalescePartitions.enabled", "true", "SPARK_DEFAULT",
                        "Lets AQE merge small post-shuffle partitions at runtime rather than living with the static count above."),
        Recommendation("spark.sql.adaptive.coalescePartitions.parallelismFirst", "false", "HEURISTIC",
                        "Default is true, which makes AQE ignore advisoryPartitionSizeInBytes in favour of maximum parallelism. "
                        "Setting false makes coalescing respect the target size below — more predictable partition sizes, "
                        "recommended once you're past initial exploration."),
        Recommendation("spark.sql.adaptive.advisoryPartitionSizeInBytes", str(target_bytes), "HEURISTIC",
                        f"Spark default is 64 MB; raised to {target_partition_mb} MB to match the read-side maxPartitionBytes target "
                        "so partition sizes are consistent across the read and shuffle boundary."),
        Recommendation("spark.sql.adaptive.skewJoin.enabled", "true", "SPARK_DEFAULT",
                        "On by default; left on."),
    ]


def recommend_read_partitioning(avg_file_size_bytes: Optional[int], target_partition_mb: int = 128) -> List[Recommendation]:
    recs = []
    target_bytes = target_partition_mb * 1024 * 1024
    recs.append(Recommendation("spark.sql.files.maxPartitionBytes", str(target_bytes), "HEURISTIC",
                f"Spark default is 128 MB; set explicitly to {target_partition_mb} MB to keep read-side and shuffle-side "
                "partition sizing consistent (see advisoryPartitionSizeInBytes above)."))
    if avg_file_size_bytes is not None:
        avg_mb = avg_file_size_bytes / (1024 * 1024)
        if avg_mb < 16:
            recs.append(Recommendation("spark.sql.files.openCostInBytes", "4194304", "HEURISTIC",
                        f"Average source file size is only {avg_mb:.1f} MB (the 'small files problem'). Spark default "
                        "openCostInBytes (4 MB) already estimates the fixed cost of opening a file, which packs multiple "
                        "small files into one partition/task rather than one task per tiny file — confirmed left at default, "
                        "flagged here because it's the setting that matters most for this data shape."))
            recs.append(Recommendation("_advisory_small_files", "n/a", "HEURISTIC",
                        f"{avg_mb:.1f} MB average file size is well under the {target_partition_mb} MB target. Consider a "
                        "compaction/OPTIMIZE pass upstream (e.g. Delta OPTIMIZE) rather than only compensating for it at read time — "
                        "many small files also inflate driver-side listing time and metadata overhead independent of Spark's partitioning."))
    return recs


def recommend_executor_shape(cluster: ClusterProfile, cache_heavy: bool = False) -> (List[Recommendation], Optional[float]):
    """Returns (recommendations, resolved_executor_memory_gb). The second value feeds
    the memory-overhead calculation and is None when it can't be resolved (e.g. Fabric,
    where overhead isn't a user-facing dial anyway)."""
    recs = []
    executor_memory_gb = None
    if cluster.platform == "fabric":
        recs.append(Recommendation("_fabric_node_size", cluster.node_size or "unspecified", "FABRIC_DOC",
                    f"Fabric ties one executor to one node (1:1), always. There is no 'cores per executor' packing "
                    f"decision the way there is on YARN — the node size *is* the executor size. "
                    f"{cluster.node_size} = {cluster.node_vcores} vCores / {cluster.node_memory_gb} GB per executor "
                    "(Fabric Spark pools are memory-optimized at a fixed 8 GB per vCore)."))
        recs.append(Recommendation("_fabric_executor_cores_note", "n/a", "FABRIC_DOC",
                    "Within a chosen node size, Fabric environments let you under-populate the executor's core count "
                    "(e.g. run an 8-vCore node's executor at 4 cores) to trade parallelism for more memory headroom per "
                    "task — set via the environment's Compute pane or the %%configure magic, not via spark.executor.cores "
                    "alone, since the node's total vCores are still reserved either way."))
    else:
        # Classic YARN-style guidance (Cloudera): avoid 1-core "thin" executors (no benefit from
        # broadcast-variable/JVM reuse across tasks) and avoid >5-core "fat" executors (HDFS client
        # concurrency and GC pause problems dominate beyond ~5 concurrent tasks per JVM).
        cores_per_executor = 4 if cache_heavy else 5
        cache_note = (" Reduced to 4 here because cache_heavy=True leaves more headroom per task for storage memory pressure."
                       if cache_heavy else " Left at the standard 5 (cache_heavy=False, so no extra per-task memory headroom was traded away).")
        recs.append(Recommendation("spark.executor.cores", str(cores_per_executor), "HEURISTIC",
                    "Classic YARN sizing guidance (Cloudera): stay at or below ~5 concurrent tasks per executor JVM. "
                    "Below ~3 cores, per-executor JVM/broadcast overhead is paid too many times; above ~5, HDFS client "
                    "concurrency and GC pause times start to dominate." + cache_note))
        if cluster.node_vcores and cluster.node_memory_gb:
            executors_per_node = max(1, (cluster.node_vcores - 1) // cores_per_executor)  # -1 core reserved for OS/NM daemon
            recs.append(Recommendation("_executors_per_node", str(executors_per_node), "HEURISTIC",
                        f"({cluster.node_vcores} vCores \u2212 1 reserved for OS/node-manager daemons) \u00f7 {cores_per_executor} cores/executor."))
            usable_node_memory_gb = max(cluster.node_memory_gb - 1, 1)  # ~1GB reserved for OS
            executor_memory_gb = usable_node_memory_gb / executors_per_node
            recs.append(Recommendation("spark.executor.memory", f"{executor_memory_gb:.2f}g", "HEURISTIC",
                        f"({cluster.node_memory_gb} GB node \u2212 ~1 GB OS reserve) \u00f7 {executors_per_node} executors/node. "
                        "This is memory BEFORE the overhead deduction below — spark.executor.memory sets JVM heap size "
                        "directly; memoryOverhead is requested by Spark on top of this, not carved out of it."))
    return recs, executor_memory_gb


def recommend_memory_fractions(cache_heavy: bool = False) -> List[Recommendation]:
    if cache_heavy:
        return [
            Recommendation("spark.memory.fraction", "0.6", "SPARK_DEFAULT", "Left at Spark default."),
            Recommendation("spark.memory.storageFraction", "0.6", "HEURISTIC",
                        "Raised from the 0.5 default because this workload is cache-heavy (repeated reuse of a persisted "
                        "DataFrame/table). This raises the floor protected from eviction by execution memory; execution "
                        "can still borrow spare storage memory when it's genuinely idle."),
        ]
    return [
        Recommendation("spark.memory.fraction", "0.6", "SPARK_DEFAULT",
                    "Spark default. Only worth raising (e.g. toward 0.7-0.8) for execution-heavy, cache-light SQL/DataFrame "
                    "jobs, and only after confirming via the Spark UI that GC time or shuffle spill — not caching — is the "
                    "actual bottleneck."),
        Recommendation("spark.memory.storageFraction", "0.5", "SPARK_DEFAULT", "Spark default; no caching signal to justify changing it."),
    ]


def recommend_executor_memory_overhead(executor_memory_gb: float, platform: str) -> Recommendation:
    if platform == "fabric":
        return Recommendation("_memory_overhead_note", "n/a", "FABRIC_DOC",
                    "Fabric's environment Compute pane exposes a fixed menu of executor-memory choices per node size "
                    "(e.g. specific GB values for a Large node) that already reserve overhead — there is no separate "
                    "spark.executor.memoryOverhead dial to set by hand as there is on generic YARN/Kubernetes Spark.")
    overhead_gb = max(0.375, 0.10 * executor_memory_gb)  # 384 MB floor, 10% factor — Spark default formula
    return Recommendation("spark.executor.memoryOverhead", f"{overhead_gb:.2f}g", "SPARK_DEFAULT",
                f"Spark default formula: max(384 MB, 0.10 \u00d7 executor memory). Covers JVM internals, PySpark worker "
                "processes, and native/off-heap allocations outside the JVM heap.")


def recommend_broadcast_threshold(
    smallest_join_side_bytes: Optional[int],
    driver_memory_gb: Optional[float] = None,
) -> Recommendation:
    default_bytes = 10 * 1024 * 1024
    if smallest_join_side_bytes is None:
        return Recommendation("spark.sql.autoBroadcastJoinThreshold", str(default_bytes), "SPARK_DEFAULT",
                    "No join-side size profile supplied, so left at Spark's conservative 10 MB default. If you know one "
                    "side of your main join is small, re-run analyze() pointed at that table to get a tuned value.")
    # Industry-common production ceiling: 200-500MB, well below typical executor memory,
    # and only if it comfortably covers the actual smallest side with headroom.
    headroom_target = int(smallest_join_side_bytes * 1.5)
    ceiling = 512 * 1024 * 1024
    recommended = min(max(headroom_target, default_bytes), ceiling)
    return Recommendation("spark.sql.autoBroadcastJoinThreshold", str(recommended), "HEURISTIC",
                f"Smallest known join side is ~{smallest_join_side_bytes/1e6:.1f} MB; recommending "
                f"{recommended/1e6:.0f} MB (1.5\u00d7 headroom over that side, capped at 512 MB). Broadcasting avoids a "
                "shuffle entirely for this join, but every executor pays this memory cost, and the driver has to collect "
                "and broadcast it first — verify spark.driver.memory has headroom.")


def recommend_autoscale_bounds(num_shuffle_partitions: int, cluster: ClusterProfile) -> List[Recommendation]:
    if cluster.platform != "fabric" or not cluster.node_vcores:
        return []
    ideal_executors = max(1, math.ceil(num_shuffle_partitions / cluster.node_vcores))
    max_nodes = ideal_executors + 1  # +1 for the driver node
    return [
        Recommendation("_autoscale_min_nodes", "1", "HEURISTIC",
                    "Start autoscale at 1 node; Efficient Scaledown (see Part 5-8 of the Efficient Scaledown internals "
                    "doc) means idle executors are released quickly, so a low floor costs little."),
        Recommendation("_autoscale_max_nodes", str(min(max_nodes, 64)), "HEURISTIC",
                    f"Estimated {ideal_executors} executors needed to give every shuffle-stage partition a core "
                    f"({num_shuffle_partitions} partitions \u00f7 {cluster.node_vcores} vCores/node), +1 node for the "
                    "driver. Capped at 64 nodes as a sanity ceiling — validate against your capacity SKU's max node limit."),
    ]


def recommend_fabric_efficient_scaledown() -> List[Recommendation]:
    """Directly reuses the recommended configuration block from the companion
    'Efficient Scaledown & Remote Shuffle Manager' internals document (Part 9.1)."""
    return [
        Recommendation("spark.remote.shuffle.enabled", "true", "FABRIC_DOC",
                    "Enables Remote Shuffle Manager. Requires NEE, Runtime 1.3+, and a non-HNS BlockBlobStorage account "
                    "reachable without Private Link — see the Efficient Scaledown internals doc, Part 10, before enabling in a "
                    "network-restricted environment."),
        Recommendation("spark.sql.rsm.decisionlayer.enabled.level", "stage", "FABRIC_DOC", "Per-stage local/remote shuffle routing."),
        Recommendation("spark.sql.adaptive.shuffleWrite.enabled", "true", "FABRIC_DOC", "AQE participates in the shuffle write phase."),
        Recommendation("spark.storage.decommission.shuffleBlocks.enabled", "true", "FABRIC_DOC", "Migrates local shuffle blocks off a decommissioning executor."),
        Recommendation("spark.storage.decommission.shuffleBlocks.cleanup", "true", "FABRIC_DOC", "Cleans up source blocks after a successful migration."),
        Recommendation("spark.storage.decommission.shuffleBlocks.migrateToFallbackStorage", "true", "FABRIC_DOC", "Falls back to Blob Storage if no peer executor can accept a migrating block."),
        Recommendation("spark.storage.decommission.fallbackStorage.cleanUp", "true", "FABRIC_DOC", "Bounds fallback storage cost over time."),
    ]


def build_full_recommendation(
    data: DataProfile,
    cluster: ClusterProfile,
    workload: WorkloadHints,
    target_partition_mb: int = 128,
    enable_efficient_scaledown: bool = False,
) -> List[Recommendation]:
    recs: List[Recommendation] = []
    recs.append(estimate_shuffle_partitions(data.total_bytes, cluster.total_executor_vcores, target_partition_mb))
    recs.extend(recommend_advisory_partition_size(target_partition_mb))
    recs.extend(recommend_read_partitioning(data.avg_file_size_bytes, target_partition_mb))
    shape_recs, resolved_executor_memory_gb = recommend_executor_shape(cluster, cache_heavy=workload.cache_heavy)
    recs.extend(shape_recs)
    recs.extend(recommend_memory_fractions(cache_heavy=workload.cache_heavy))
    if cluster.platform == "fabric" and cluster.node_memory_gb:
        recs.append(recommend_executor_memory_overhead(cluster.node_memory_gb, cluster.platform))
    elif resolved_executor_memory_gb:
        recs.append(recommend_executor_memory_overhead(resolved_executor_memory_gb, cluster.platform))
    recs.append(recommend_broadcast_threshold(data.smallest_join_side_bytes, cluster.driver_memory_gb))
    shuffle_rec = recs[0]
    recs.extend(recommend_autoscale_bounds(int(shuffle_rec.value), cluster))
    if data.expected_skew:
        recs.append(Recommendation("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5", "SPARK_DEFAULT",
                    "Left at default; skew handling is already enabled above. If skew is severe, inspect the Spark UI "
                    "stage detail for the actual skew ratio before overriding this."))
    if cluster.platform == "fabric" and enable_efficient_scaledown:
        recs.extend(recommend_fabric_efficient_scaledown())
    return recs

### Quick self-test of the heuristic engine

Runs a handful of the same sanity checks used to validate this module during development —
no Spark session required. If any of these fail, something in the cell above was edited in a
way that broke the arithmetic; fix that before trusting anything downstream.

In [3]:
# Lightweight self-test — mirrors the fuller suite in test_spark_autoconfig.py
_d = DataProfile(total_bytes=int(15 * 1024**3), file_count=120)
_c = ClusterProfile(platform="fabric", node_size="Medium", num_nodes=4)
_w = WorkloadHints(kind="batch_etl")
_recs = build_full_recommendation(_d, _c, _w)
assert any(r.key == "spark.sql.shuffle.partitions" for r in _recs)
assert int(_recs[0].value) > 0

assert nearest_fabric_node(10)["name"] == "Large"
assert nearest_fabric_node(4)["name"] == "Small"

_tiny = estimate_shuffle_partitions(total_input_bytes=10 * 1024 * 1024, total_executor_vcores=200)
assert int(_tiny.value) == 400, "parallelism floor should dominate for tiny data on a big cluster"

_r_broadcast = recommend_broadcast_threshold(None)
assert _r_broadcast.value == str(10 * 1024 * 1024)

# scope classification — the empirically-verified split from Part 2
assert _scope_for_key("spark.sql.shuffle.partitions") == "runtime"
assert _scope_for_key("spark.executor.cores") == "session_start"
assert _scope_for_key("spark.memory.fraction") == "session_start"
assert _scope_for_key("spark.remote.shuffle.enabled") == "session_start"

print("Self-test passed:", len(_recs), "recommendations generated for the sample scenario.")
del _d, _c, _w, _recs, _tiny, _r_broadcast

Self-test passed: 15 recommendations generated for the sample scenario.


---
## Part 2 — The Spark/Fabric-facing wrapper (`SparkAutoConfigurator`)

This is where the pure heuristics from Part 1 meet a live `SparkSession`: reading real data
characteristics, best-effort cluster introspection, and applying the result.

### The one finding worth reading before the code: not every Spark config is runtime-mutable

While building this notebook, calling `spark.conf.set("spark.executor.cores", "5")` against a
**live, already-started** local Spark 3.5.1 session raised:

```
pyspark.errors.exceptions.captured.AnalysisException: [CANNOT_MODIFY_CONFIG]
Cannot modify the value of the Spark config: "spark.executor.cores"
```

Probing every setting this notebook recommends against that same live session (`probe_mutability.py`,
included in the companion files) produced a clean split:

**Runtime-mutable** (safe via `spark.conf.set()` on a session that's already running):
`spark.sql.shuffle.partitions`, every `spark.sql.adaptive.*` key, `spark.sql.files.maxPartitionBytes`,
`spark.sql.files.openCostInBytes`, `spark.sql.autoBroadcastJoinThreshold`.

**Session-start-only** (raises `CANNOT_MODIFY_CONFIG` if you try mid-session):
`spark.executor.cores`, `spark.executor.memory`, `spark.executor.memoryOverhead`,
`spark.memory.fraction`, `spark.memory.storageFraction`, `spark.driver.memory`, every
`spark.dynamicAllocation.*` key, and (architecturally, even where a given Spark build doesn't
enforce it) the shuffle-plugin wiring behind Remote Shuffle Manager / Shuffle Migration /
Decision Layer — those are resolved once when `SparkEnv` is constructed.

This matches, independently, what the Fabric community's own troubleshooting advice for
executor OOM errors already does in practice: it fixes `spark.executor.memoryOverhead` and
`spark.executor.memory` via the **`%%configure` magic cell**, not `spark.conf.set()` — because
`%%configure` runs *before* the session starts. `SparkAutoConfigurator.apply()` below only ever
touches the runtime-mutable set; `session_start_snippet()` renders the rest as a ready-to-paste
`%%configure` block instead of quietly producing a config no one can use.

In [4]:
"""
spark_autoconfig.py
====================
The Spark- and Fabric-facing utility class. Wraps spark_autoconfig_core's pure heuristic
engine with real I/O: reading data characteristics from a live SparkSession, best-effort
cluster-shape introspection, applying the resulting configuration, and printing a report.

Design principle: every piece of live introspection here is wrapped in try/except with an
explicit, user-overridable fallback. Nothing in this file invents or assumes an undocumented
API surface — where Fabric doesn't expose something through a stable, documented call
(e.g. "what node size is this pool"), the utility asks for it as a parameter instead of
guessing.
"""

# NOTE: DataProfile, ClusterProfile, WorkloadHints, Recommendation, build_full_recommendation
# and FABRIC_NODE_SIZES are already defined in this notebook's namespace from the Part 1 cell
# above. This `import` line is here in the standalone spark_autoconfig.py file (for reuse as
# a real importable module elsewhere) but is a no-op to keep in a notebook where Part 1 already ran.
from typing import Optional, List


class SparkAutoConfigurator:
    def __init__(self, spark, platform: Optional[str] = None, verbose: bool = True):
        self.spark = spark
        self.verbose = verbose
        self.platform = platform or self._detect_platform()
        self._last_data_profile: Optional[DataProfile] = None
        self._last_cluster_profile: Optional[ClusterProfile] = None
        self._last_recs: Optional[List[Recommendation]] = None

    # ---------------------------------------------------------------- platform / cluster

    def _detect_platform(self) -> str:
        """Best-effort only. Fabric notebooks have `notebookutils` injected into the global
        namespace at runtime; its mere presence is a reasonable signal, but this is a
        convenience heuristic, not a documented detection API — always fine to override
        with SparkAutoConfigurator(spark, platform='fabric'|'oss')."""
        try:
            import notebookutils  # noqa: F401
            return "fabric"
        except ImportError:
            return "oss"

    def detect_cluster(
        self,
        node_size: Optional[str] = None,
        num_nodes: Optional[int] = None,
        node_vcores: Optional[int] = None,
        node_memory_gb: Optional[float] = None,
    ) -> ClusterProfile:
        """
        Explicit parameters always win. Where not supplied, falls back to best-effort live
        introspection via the Spark status tracker and spark.conf — both real, if
        underscore-prefixed / internal, PySpark patterns; wrapped defensively because they
        are not part of PySpark's stable public API and can differ across versions/platforms.
        """
        live_executors, live_cores, live_mem_gb = self._introspect_live_cluster()

        if self.platform == "fabric":
            resolved_num_nodes = num_nodes if num_nodes is not None else (live_executors + 1 if live_executors else 1)
            cp = ClusterProfile(platform="fabric", node_size=node_size or "Medium", num_nodes=resolved_num_nodes)
            if node_size is None and self.verbose:
                print("[detect_cluster] No node_size supplied — defaulted to 'Medium'. "
                      "Fabric does not expose pool node size through a documented notebookutils/spark.conf call, "
                      "so pass node_size explicitly (Small/Medium/Large/XLarge/XXLarge) for an accurate recommendation.")
        else:
            cp = ClusterProfile(
                platform="oss",
                num_nodes=num_nodes if num_nodes is not None else max(live_executors + 1, 2),
                node_vcores=node_vcores or live_cores or 8,
                node_memory_gb=node_memory_gb or live_mem_gb or 32.0,
                driver_memory_gb=self._get_driver_memory_gb(),
            )
        self._last_cluster_profile = cp
        return cp

    def _introspect_live_cluster(self):
        """Returns (num_live_executors, cores_per_executor, memory_gb_per_executor) or
        (None, None, None) on any failure. Uses SparkContext's status tracker (a real,
        commonly-used pattern for live executor counts) and spark.conf for the rest."""
        try:
            sc = self.spark.sparkContext
            infos = sc._jsc.sc().statusTracker().getExecutorInfos()
            num_executors = max(len(infos) - 1, 1)  # one entry is the driver
        except Exception:
            num_executors = None
        try:
            cores = int(self.spark.conf.get("spark.executor.cores"))
        except Exception:
            cores = None
        try:
            mem_str = self.spark.conf.get("spark.executor.memory")  # e.g. "8g"
            mem_gb = float(mem_str.lower().replace("g", "").replace("m", "")) if mem_str else None
            if mem_str and "m" in mem_str.lower():
                mem_gb = mem_gb / 1024
        except Exception:
            mem_gb = None
        return num_executors, cores, mem_gb

    def _get_driver_memory_gb(self) -> Optional[float]:
        try:
            mem_str = self.spark.conf.get("spark.driver.memory")
            val = float(mem_str.lower().replace("g", "").replace("m", ""))
            return val / 1024 if "m" in mem_str.lower() else val
        except Exception:
            return None

    # ---------------------------------------------------------------------- data profiling

    def analyze(
        self,
        path: str,
        table_format: str = "delta",
        smallest_join_side_path: Optional[str] = None,
        smallest_join_side_format: str = "delta",
        expected_skew: bool = False,
        estimate_row_count: bool = False,
    ) -> DataProfile:
        """
        Profiles the data at `path`. Tries, in order:
          1. DESCRIBE DETAIL (Delta only) — exact sizeInBytes/numFiles from the Delta log,
             no data scan required. Cheapest and most accurate path for Lakehouse tables.
          2. df.inputFiles() + Hadoop FileSystem file-status lookup — works for any
             Spark-readable format, at the cost of listing every file individually.
          3. Raises, asking the caller to construct a DataProfile manually instead.
        """
        total_bytes, file_count = self._get_size_and_files(path, table_format)
        if self.verbose:
            print(f"[analyze] {path}: {total_bytes/1e9:.2f} GB across {file_count} files.")

        row_count = None
        if estimate_row_count:
            try:
                row_count = self.spark.read.format(table_format).load(path).count()
            except Exception as e:
                if self.verbose:
                    print(f"[analyze] row count estimate skipped ({e!r}).")

        join_bytes = None
        if smallest_join_side_path:
            join_bytes, _ = self._get_size_and_files(smallest_join_side_path, smallest_join_side_format)

        profile = DataProfile(
            total_bytes=total_bytes, file_count=max(file_count, 1),
            estimated_row_count=row_count, smallest_join_side_bytes=join_bytes,
            expected_skew=expected_skew, source_format=table_format,
        )
        self._last_data_profile = profile
        return profile

    def _get_size_and_files(self, path: str, table_format: str):
        """Delta DESCRIBE DETAIL when possible (cheap, exact), else a Hadoop FS listing scan."""
        if table_format == "delta":
            try:
                row = self.spark.sql(f"DESCRIBE DETAIL delta.`{path}`").collect()[0]
                return int(row["sizeInBytes"]), int(row["numFiles"])
            except Exception:
                pass
        return self._scan_via_hadoop_fs(path, table_format)

    def _scan_via_hadoop_fs(self, path: str, table_format: str):
        """Falls back to Spark's own Hadoop FileSystem bridge to sum real file sizes.
        Uses spark._jsc / spark._jvm — real, working, widely-used PySpark patterns for this
        exact purpose, but internal/underscore-prefixed, so wrapped defensively."""
        df = self.spark.read.format(table_format).load(path)
        input_files = df.inputFiles()
        if not input_files:
            raise RuntimeError(f"No input files resolved for path: {path}")
        hconf = self.spark._jsc.hadoopConfiguration()
        jvm = self.spark._jvm
        total = 0
        for f in input_files:
            jpath = jvm.org.apache.hadoop.fs.Path(f)
            fs = jpath.getFileSystem(hconf)
            total += fs.getFileStatus(jpath).getLen()
        if self.verbose:
            print(f"[analyze] Hadoop FS scan: {total/1e9:.2f} GB across {len(input_files)} files.")
        return total, len(input_files)

    # ---------------------------------------------------------------------- recommend / apply

    def recommend(
        self,
        data: Optional[DataProfile] = None,
        cluster: Optional[ClusterProfile] = None,
        workload: str = "batch_etl",
        cache_heavy: bool = False,
        target_partition_mb: int = 128,
        enable_efficient_scaledown: bool = False,
    ) -> List[Recommendation]:
        data = data or self._last_data_profile
        cluster = cluster or self._last_cluster_profile
        if data is None:
            raise ValueError("No DataProfile available — call analyze(path) first, or pass data= explicitly.")
        if cluster is None:
            raise ValueError("No ClusterProfile available — call detect_cluster() first, or pass cluster= explicitly.")
        hints = WorkloadHints(kind=workload, cache_heavy=cache_heavy)
        recs = build_full_recommendation(
            data, cluster, hints,
            target_partition_mb=target_partition_mb,
            enable_efficient_scaledown=(enable_efficient_scaledown and cluster.platform == "fabric"),
        )
        self._last_recs = recs
        return recs

    def apply(self, recs: Optional[List[Recommendation]] = None, dry_run: bool = False) -> dict:
        """
        Applies only the configs that are genuinely safe to set on a live session
        (Recommendation.scope == 'runtime', e.g. every spark.sql.* / AQE / broadcast-threshold
        key). Session-start-only configs (executor cores/memory, spark.memory.*,
        dynamicAllocation.*, and shuffle-plugin wiring) are NEVER passed to spark.conf.set —
        doing so raises AnalysisException[CANNOT_MODIFY_CONFIG] on a real cluster, verified
        empirically against a live Spark 3.5.1 session. Use session_start_snippet() to get
        those as a %%configure block (Fabric) or SparkConf snippet (OSS) to apply *before*
        the session starts.
        """
        recs = recs or self._last_recs
        if recs is None:
            raise ValueError("No recommendations to apply — call recommend() first.")
        applied = {}
        skipped_session_start = []
        for r in recs:
            if r.key.startswith("_"):
                continue
            if r.scope == "session_start":
                skipped_session_start.append(r.key)
                continue
            if dry_run:
                if self.verbose:
                    print(f"[dry-run] would set {r.key} = {r.value}")
            else:
                self.spark.conf.set(r.key, r.value)
                if self.verbose:
                    print(f"[apply] {r.key} = {r.value}")
            applied[r.key] = r.value
        if skipped_session_start and self.verbose:
            print(f"\n[apply] Skipped {len(skipped_session_start)} session-start-only config(s) — "
                  "these cannot be changed on a running session. Call session_start_snippet() to "
                  "get them as a %%configure block (Fabric) or SparkConf snippet (OSS) for next "
                  "session start:")
            for k in skipped_session_start:
                print(f"           - {k}")
        return applied

    def session_start_snippet(self, recs: Optional[List[Recommendation]] = None, as_format: str = "fabric") -> str:
        """Renders session-start-only recommendations as something directly usable:
        - as_format='fabric': a %%configure -f JSON cell (paste as the notebook's FIRST cell,
          before any Spark code runs — %%configure must run before the session starts).
        - as_format='sparkconf': a SparkConf(...).set(...) chain for building the session yourself.
        """
        recs = recs or self._last_recs
        if recs is None:
            raise ValueError("No recommendations available — call recommend() first.")
        session_recs = [r for r in recs if r.scope == "session_start" and not r.key.startswith("_")]
        if not session_recs:
            return "(no session-start-only configs in this recommendation set)"
        if as_format == "fabric":
            import json
            conf_block = {r.key: r.value for r in session_recs}
            body = {"conf": conf_block}
            return "%%configure -f\n" + json.dumps(body, indent=2)
        lines = ["conf = SparkConf()"]
        for r in session_recs:
            lines.append(f'conf.set("{r.key}", "{r.value}")')
        lines.append("spark = SparkSession.builder.config(conf=conf).getOrCreate()")
        return "\n".join(lines)

    def report(self, recs: Optional[List[Recommendation]] = None) -> str:
        recs = recs or self._last_recs
        if recs is None:
            raise ValueError("No recommendations to report — call recommend() first.")
        lines = ["=" * 100, "SPARK AUTO-CONFIG REPORT", "=" * 100]
        if self._last_data_profile:
            d = self._last_data_profile
            lines.append(f"Data:    {d.total_gb:.2f} GB across {d.file_count} files "
                          f"(avg {d.avg_file_size_bytes/1e6:.1f} MB/file)"
                          + (f", ~{d.estimated_row_count:,} rows" if d.estimated_row_count else ""))
        if self._last_cluster_profile:
            c = self._last_cluster_profile
            if c.platform == "fabric":
                lines.append(f"Cluster: Fabric, node_size={c.node_size}, num_nodes={c.num_nodes} "
                              f"({c.num_executors} executors x {c.node_vcores} vCores/{c.node_memory_gb} GB)")
            else:
                lines.append(f"Cluster: OSS/YARN, num_nodes={c.num_nodes}, "
                              f"{c.node_vcores} vCores/{c.node_memory_gb} GB per node")
        lines.append("-" * 100)
        runtime_recs = [r for r in recs if r.scope == "runtime" and not r.key.startswith("_")]
        session_recs = [r for r in recs if r.scope == "session_start" and not r.key.startswith("_")]
        note_recs = [r for r in recs if r.key.startswith("_")]

        def render(rec_list):
            for r in rec_list:
                lines.append(f"[{r.basis:13s}] {r.key}")
                lines.append(f"{'':17s}= {r.value}")
                lines.append(f"{'':17s}{r.reason}")
                lines.append("")

        lines.append(f"RUNTIME-MUTABLE — safe to apply on the current session via apply() ({len(runtime_recs)}):")
        render(runtime_recs)
        lines.append(f"SESSION-START-ONLY — needs a %%configure block / new session ({len(session_recs)}):")
        lines.append("(get these via session_start_snippet() — spark.conf.set() will raise CANNOT_MODIFY_CONFIG)")
        lines.append("")
        render(session_recs)
        if note_recs:
            lines.append(f"NOTES ({len(note_recs)}):")
            for r in note_recs:
                lines.append(f"[{r.basis:13s}] (note) {r.key[1:]}")
                lines.append(f"{'':17s}{r.reason}")
                lines.append("")
        lines.append("=" * 100)
        lines.append("Legend: SPARK_DEFAULT = documented Apache Spark default | FABRIC_DOC = documented Fabric "
                      "behaviour/figure | HEURISTIC = community/industry rule of thumb — validate against the Spark UI.")
        text = "\n".join(lines)
        if self.verbose:
            print(text)
        return text

---
## Part 3 — Live demo, against real (synthetic) data

Everything below runs against an **actual local SparkSession** — not a mock. It creates a small
synthetic Parquet dataset on local disk, then runs the full
`analyze → detect_cluster → recommend → report → apply` pipeline against it, so every line
of output on this page is real, not illustrative.

If a `spark` variable already exists in this notebook's runtime (true inside Fabric, Databricks,
and most managed notebook environments), it's reused as-is; otherwise a local standalone session
is created so this still runs anywhere.

In [5]:
# --- Session: Fabric is the default target -------------------------------------
# In Fabric you do NOT create a Spark session. The Livy layer starts it before your first
# cell runs, and `spark` (plus `sc`, `notebookutils`) are already bound. Calling
# SparkSession.builder there is at best a no-op via getOrCreate() and at worst misleading:
# master(), Delta wiring and executor shape are all decided by the Environment/pool, not here.
#
# Session-start settings belong in a %%configure -f cell ABOVE this one, or in the
# Environment's Spark properties. Only runtime-mutable keys can be set from code.
try:
    spark                      # noqa: F821  <- Fabric (and any live session): already provided
    IN_FABRIC = True
except NameError:
    # Local/dev fallback ONLY. Never runs in Fabric.
    IN_FABRIC = False
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    _b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
          .config("spark.driver.memory", "2g")
          .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
          .config("spark.sql.catalog.spark_catalog",
                  "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(_b).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(("Fabric session (provided)" if IN_FABRIC else "local session (dev fallback)"),
      "| Spark", spark.version)


26/08/03 08:26:32 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/03 08:26:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/03 08:26:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


No existing session found — created a local standalone SparkSession for this demo.


In [6]:
# Build a small synthetic dataset to analyze. Parquet (not Delta) is used here purely so this
# cell has zero extra dependencies and runs anywhere; SparkAutoConfigurator.analyze() supports
# both — see the DESCRIBE DETAIL fast path in Part 2 for why Delta tables (the norm in a Fabric
# Lakehouse) profile faster than this Parquet fallback path does.
import shutil
from pyspark.sql import functions as F

DEMO_PATH = "/tmp/spark_autoconfig_demo/orders_fact"
DEMO_JOIN_PATH = "/tmp/spark_autoconfig_demo/region_dim"
shutil.rmtree("/tmp/spark_autoconfig_demo", ignore_errors=True)

(
    spark.range(0, 4_000_000)
    .withColumn("amount", (F.col("id") % 997).cast("double"))
    .withColumn("region_id", (F.col("id") % 12).cast("int"))
    .repartition(150)
    .write.format("parquet").mode("overwrite").save(DEMO_PATH)
)
spark.range(0, 12).withColumnRenamed("id", "region_id").write.format("parquet").mode("overwrite").save(DEMO_JOIN_PATH)
print("Synthetic demo data written.")

Synthetic demo data written.


In [7]:
from spark_autoconfig import SparkAutoConfigurator

configurator = SparkAutoConfigurator(spark, platform=PLATFORM)

data_profile = configurator.analyze(
    DEMO_PATH,
    table_format="parquet",
    smallest_join_side_path=DEMO_JOIN_PATH,
    smallest_join_side_format="parquet",
    expected_skew=False,
    estimate_row_count=True,
)
data_profile

[analyze] Hadoop FS scan: 0.03 GB across 150 files.
[analyze] /tmp/spark_autoconfig_demo/orders_fact: 0.03 GB across 150 files.


[analyze] Hadoop FS scan: 0.00 GB across 4 files.


DataProfile(total_bytes=27604328, file_count=150, avg_file_size_bytes=184028, estimated_row_count=4000000, smallest_join_side_bytes=2023, expected_skew=False, source_format='parquet')

In [8]:
# Cluster shape: on Fabric, pass your pool's actual node_size/num_nodes (not reliably
# auto-detectable — see Part 2's detect_cluster() docstring for why). Standalone, it falls back
# to introspecting the live local session.
cluster_profile = configurator.detect_cluster(
    node_size="Medium", num_nodes=4,          # used if PLATFORM == "fabric"
    node_vcores=4, node_memory_gb=8,           # used if PLATFORM == "oss" and not auto-detected
)
cluster_profile

ClusterProfile(platform='oss', node_size=None, num_nodes=4, node_vcores=4, node_memory_gb=8, driver_memory_gb=2.0)

In [9]:
recs = configurator.recommend(
    workload="batch_etl",
    cache_heavy=False,
    target_partition_mb=128,
    enable_efficient_scaledown=(PLATFORM == "fabric"),
)
report_text = configurator.report()

SPARK AUTO-CONFIG REPORT
Data:    0.03 GB across 150 files (avg 0.2 MB/file), ~4,000,000 rows
Cluster: OSS/YARN, num_nodes=4, 4 vCores/8 GB per node
----------------------------------------------------------------------------------------------------
RUNTIME-MUTABLE — safe to apply on the current session via apply() (9):
[HEURISTIC    ] spark.sql.shuffle.partitions
                 = 24
                 max(size-based [0.03 GB ÷ 128 MB → 1], parallelism floor [12 vCores × 2 → 24])

[SPARK_DEFAULT] spark.sql.adaptive.enabled
                 = true
                 On by default since Spark 3.2 / all current Fabric runtimes; set explicitly for clarity.

[SPARK_DEFAULT] spark.sql.adaptive.coalescePartitions.enabled
                 = true
                 Lets AQE merge small post-shuffle partitions at runtime rather than living with the static count above.

[HEURISTIC    ] spark.sql.adaptive.coalescePartitions.parallelismFirst
                 = false
                 Default is true, wh

In [10]:
applied = configurator.apply(dry_run=False)
print("\nRuntime configs applied to the live session:")
for k, v in applied.items():
    print(f"  {k} = {v}   (live value: {spark.conf.get(k)})")

[apply] spark.sql.shuffle.partitions = 24
[apply] spark.sql.adaptive.enabled = true
[apply] spark.sql.adaptive.coalescePartitions.enabled = true
[apply] spark.sql.adaptive.coalescePartitions.parallelismFirst = false
[apply] spark.sql.adaptive.advisoryPartitionSizeInBytes = 134217728
[apply] spark.sql.adaptive.skewJoin.enabled = true
[apply] spark.sql.files.maxPartitionBytes = 134217728
[apply] spark.sql.files.openCostInBytes = 4194304
[apply] spark.sql.autoBroadcastJoinThreshold = 10485760

[apply] Skipped 5 session-start-only config(s) — these cannot be changed on a running session. Call session_start_snippet() to get them as a %%configure block (Fabric) or SparkConf snippet (OSS) for next session start:
           - spark.executor.cores
           - spark.executor.memory
           - spark.memory.fraction
           - spark.memory.storageFraction
           - spark.executor.memoryOverhead

Runtime configs applied to the live session:
  spark.sql.shuffle.partitions = 24   (live value:

In [11]:
print(configurator.session_start_snippet(as_format="fabric"))

%%configure -f
{
  "conf": {
    "spark.executor.cores": "5",
    "spark.executor.memory": "7.00g",
    "spark.memory.fraction": "0.6",
    "spark.memory.storageFraction": "0.5",
    "spark.executor.memoryOverhead": "0.70g"
  }
}


---
## Part 4 — Using this against a real Fabric Lakehouse table

The demo in Part 3 used synthetic local Parquet so this notebook runs anywhere with zero setup.
Against a real Fabric Lakehouse, three things change: the path format, the table format
(`"delta"` — enables the fast `DESCRIBE DETAIL` profiling path from Part 2), and — because
`%%configure` must be the notebook's first executable cell, before any Spark code runs — the
session-start block from `session_start_snippet()` needs to be copied into a `%%configure` cell
**above** everything else, then the notebook re-run from the top.

```python
# --- Cell 1 of a real notebook, BEFORE any other Spark code ---
# %%configure -f
# { "conf": { "spark.executor.memory": "21g", "spark.memory.fraction": "0.6", ... } }
# (paste the exact output of session_start_snippet() here, then re-run from the top)

# --- Later cell: profiling and runtime-mutable recommendations ---
from spark_autoconfig import SparkAutoConfigurator

configurator = SparkAutoConfigurator(spark, platform="fabric")

data_profile = configurator.analyze(
    "Tables/sales_fact",                       # a Lakehouse table path, or an abfss:// path
    table_format="delta",
    smallest_join_side_path="Tables/region_dim",
    smallest_join_side_format="delta",
    expected_skew=False,
    estimate_row_count=False,                  # a full count() scans the table — opt in deliberately
)

cluster_profile = configurator.detect_cluster(
    node_size="Large",                         # read this off your pool's Compute settings
    num_nodes=8,                                # your pool's current/max node count
)

recs = configurator.recommend(workload="batch_etl", enable_efficient_scaledown=True)
configurator.report()
configurator.apply()                            # applies the runtime-mutable subset only
print(configurator.session_start_snippet())     # paste this into Cell 1 next time, if it changed
```

**Why `estimate_row_count` defaults to `False`:** calling `.count()` on a large table triggers a
full scan. `DESCRIBE DETAIL`'s `sizeInBytes`/`numFiles` (used automatically for `table_format="delta"`)
costs nothing beyond a Delta log read — that's why it's the default profiling path, not a fallback.

---
## Part 5 — Configuration quick-reference

Every Spark default this notebook's heuristics reason relative to, in one place.

In [12]:
print(f"{'Setting':55s} {'Spark default':20s}")
print("-" * 75)
for k, v in SPARK_DEFAULTS.items():
    print(f"{k:55s} {v:20s}")

Setting                                                 Spark default       
---------------------------------------------------------------------------
spark.sql.files.maxPartitionBytes                       134217728           
spark.sql.shuffle.partitions                            200                 
spark.sql.adaptive.enabled                              true                
spark.sql.adaptive.coalescePartitions.enabled           true                
spark.sql.adaptive.advisoryPartitionSizeInBytes         67108864            
spark.sql.adaptive.coalescePartitions.minPartitionSize  1048576             
spark.sql.adaptive.skewJoin.enabled                     true                
spark.sql.adaptive.skewJoin.skewedPartitionFactor       5                   
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes 268435456           
spark.sql.autoBroadcastJoinThreshold                    10485760            
spark.memory.fraction                                   0.6              

In [13]:
print(f"{'Fabric node size':12s} {'vCores':8s} {'Memory (GB)':12s}")
print("-" * 34)
for n in FABRIC_NODE_SIZES:
    print(f"{n['name']:12s} {n['vcores']:<8d} {n['memory_gb']:<12d}")
print(f"\n1 Fabric Capacity Unit (CU) = {FABRIC_VCORES_PER_CU} Spark vCores.")

Fabric node size vCores   Memory (GB) 
----------------------------------
Small        4        32          
Medium       8        64          
Large        16       128         
XLarge       32       256         
XXLarge      64       512         

1 Fabric Capacity Unit (CU) = 2 Spark vCores.


---
## Limitations — read before trusting this in production

1. **Shuffle size is a proxy, not a measurement.** `estimate_shuffle_partitions()` uses total
   input bytes as a stand-in for actual shuffle write bytes. A `groupBy` that aggregates away
   95% of rows shuffles far less than its input; a row-exploding `join` shuffles more. After the
   first real run, open the Spark UI's SQL tab, find the actual "Shuffle Write" figure per stage,
   and feed a corrected `shuffle_fraction_of_input` into `estimate_shuffle_partitions()` directly
   if the default assumption (1.0) was badly wrong for your query.
2. **One data profile, one recommendation.** This models a single dominant read + shuffle
   pattern. A notebook with five very different stages (a tiny dimension scan, a 2 TB fact join,
   a small aggregation) doesn't have one "correct" `spark.sql.shuffle.partitions" — with AQE's
   coalescing on (the default here), the initial number matters less than it used to, but it
   still isn't stage-aware the way the Decision Layer's per-stage routing is (see the Efficient
   Scaledown internals doc, Part 5.3).
3. **Cluster auto-detection is best-effort.** `detect_cluster()`'s live introspection uses
   `spark.conf.get(...)` and `SparkContext`'s status tracker — real, working patterns, but not
   part of PySpark's stable public API, and Fabric doesn't expose pool node size through any
   documented call as of this writing. Always pass `node_size`/`num_nodes` explicitly for a
   recommendation you'd stake a production job on.
4. **Not skew-aware beyond a boolean flag.** `expected_skew=True` only confirms AQE's skew-join
   settings are on; it doesn't compute a skew factor from your actual key distribution. Real
   skew handling still comes from AQE at runtime (see the companion Spark Internals doc's AQE
   section) — this flag is a reminder, not a fix.
5. **Doesn't touch streaming, ML training loops, or notebook-level session sizing beyond one
   analysis.** Built for batch/ETL and ad hoc SQL-shaped workloads specifically.

## References

- Companion documents: *"Efficient Scaledown & the Remote Shuffle Manager — Internals Reference"*
  and the interactive *"Spark Internals"* HTML guide (same session).
- Apache Spark: Job Scheduling (dynamic allocation), Performance Tuning (AQE) — spark.apache.org/docs/latest
- Cloudera, *"How-to: Tune Your Apache Spark Jobs"* (executor sizing / cores-per-executor guidance)
- Microsoft Learn, *"Apache Spark compute for Data Engineering and Data Science"* (Fabric node sizes, CU→vCore ratio)
- Databricks Engineering Blog, *"Adaptive Query Execution: Speeding Up Spark SQL at Runtime"*
- Salesforce Engineering Blog, *"How to Optimize Your Apache Spark Application with Partitions"*